In [ ]:
import os
import json
import base64
import mimetypes
import time
from dotenv import load_dotenv

from pathlib import Path
from openai import OpenAI

# Load environment variables
load_dotenv()

In [ ]:
data_dir = "../data"

input_dir = os.path.join(data_dir, "2_enrich", "a_audio")
attachments_dir = os.path.join(data_dir, "1_parser", "attachments")

output_dir = os.path.join(data_dir, "2_enrich", "b_pictures")
os.makedirs(output_dir, exist_ok=True)

In [ ]:
# Initialize OpenAI client with environment variables
api_key = os.getenv("MISTRAL_API_KEY_VISION")
base_url = os.getenv("MISTRAL_BASE_URL")

if not api_key:
    raise ValueError("MISTRAL_API_KEY_VISION environment variable is not set")
if not base_url:
    raise ValueError("MISTRAL_BASE_URL environment variable is not set")

client = OpenAI(api_key=api_key, base_url=base_url)
supported_extensions = {'.png', '.jpg', '.jpeg', '.webp', '.gif'}

In [ ]:
def encode_image(image_path):
    """Encode the image to base64."""
    try:
        with open(image_path, "rb") as image_file:
            return base64.b64encode(image_file.read()).decode('utf-8')
    except FileNotFoundError:
        print(f"Error: The file {image_path} was not found.")
        return None
    except Exception as e:  # Added general exception handling
        print(f"Error: {e}")
        return None

In [ ]:
def get_mimetype_from_extension(filename):
    """Get MIME type from file extension"""
    mimetype, _ = mimetypes.guess_type(filename)
    return mimetype or 'application/octet-stream'

In [ ]:
def summarize(image_path: Path) -> str:
    base64_image = encode_image(image_path)
    if base64_image is None:
        raise RuntimeError(f'Failed to encode image: {image_path}')

    prompt = "Was ist auf diesem Bild zu sehen? Benutze maximal 4 Sätze für die Zusammenfassung in Deutsch. Falls Text zu sehen ist, transkribiere diesen. Wenn der Text in einer anderen Sprache als Deutsch ist, übersetze ihn zusätzlich in das Deutsche. Das Ausgabe Format ist Json: {\"summary\": \"<Zusammenfassung>\", \"transcription\": \"<Transkription>\", \"language\": \"<Sprache der Transkription>\", \"translation\": \"<Transkription Translated>\"} Wenn es keinen Text auf dem Bild gibt, lasse die Felder \"transcription\", \"language\" und \"translation\" leer."
    model = 'mistral-medium-latest'
    mimetype = get_mimetype_from_extension(image_path)
    print(f"Summarizing image: {image_path} with model {model} and mimetype {mimetype}")

    result = client.chat.completions.create(
        model=model,
        messages=[
            {
                "role": "system",
                "content": "Du bist ein Hilfsbereiter Assistent, welcher Bilder analysiert."
            },
            {
                "role": "user",
                "content": [
                {"type": "text", "text": prompt},
                {
                    "type": "image_url",
                    "image_url": {"url": f"data:{mimetype};base64,{base64_image}"}
                }
            ],
            }
        ],
        timeout=30,
        max_tokens=5000
    )
    print(f"Summarized image: {image_path} with model {model}")
    if not result.choices:
        raise RuntimeError(f'Failed to summarize image: {image_path}')

    content = result.choices[0].message.content
    if content is None:
        raise RuntimeError(f'No content received for image: {image_path}')
    
    return content

In [ ]:
all_files = [os.path.join(input_dir, f) for f in os.listdir(input_dir)]
for idx, file_path in enumerate(all_files, 1):
    if not file_path.endswith('.json'):
        continue
    with open(file_path, 'r', encoding='utf-8') as f:
        data = json.load(f)
    
    # print(json.dumps(data, indent=2, ensure_ascii=False))
    for msg in data.get('Messages', []):
        for attach in msg.get('Attachments', []):
            attachment_path = Path(os.path.join(attachments_dir, attach.get('path', '')))
            ext = attachment_path.suffix.lower()
            if ext in supported_extensions:
                transcript = summarize(attachment_path)
                print(f"Summarized {attachment_path}: {transcript}")
                msg['text'] = msg.get('text', '') + '<image-summary-' + str(attach['id']) + '>' + transcript + '</image-summary-' + str(attach['id']) + '>'

    file_name = os.path.basename(file_path)
    output_path = os.path.join(output_dir, file_name)
    with open(output_path, 'w', encoding='utf-8') as f:
        json.dump(data, f, ensure_ascii=False, indent=2)
    print(f"Processed {file_path} ({idx}/{len(all_files)})")